# 🌊 AquaYOLO: Automated Marine Dataset Download & YOLO Training

This notebook downloads your 3 marine datasets directly inside Google Colab (no slow Drive uploads needed!), harmonizes conflicting labels into a unified marine taxonomy, and runs high-accuracy YOLO training.

### 📌 What This Notebook Does:
1. **⚡ Fast Direct In-Colab Download**: Directly downloads at 100+ MB/s from Kaggle & Roboflow:
   - `slavkoprytula/aquarium-data-cots` (Kaggle)
   - `vencerlanz09/sea-animals-image-dataste` (Kaggle)
   - `boats-hr2s2/marine-life-pzpu1` (Roboflow)
2. **🔄 Automated Multi-Dataset Harmonizer**: Resolves class ID collisions between datasets and converts classification images into detection targets.
3. **🎯 High-Accuracy Training**: Trains YOLOv8 with transfer learning, cosine learning rate decay, and mosaic augmentation.
4. **📊 Complete Metric Graphs**: Displays all generated metric graphs directly in notebook outputs:
   - `results.png` (Training & Validation Losses, Precision, Recall, mAP@0.50, mAP@0.50:0.95)
   - `confusion_matrix.png` & `confusion_matrix_normalized.png`
   - `PR_curve.png` (Precision-Recall curve)
   - `F1_curve.png`, `P_curve.png`, `R_curve.png`
   - `val_batch0_pred.jpg` (Visual prediction comparisons)

## 🚀 Step 1: Check GPU Runtime
Make sure you are using a GPU runtime (Runtime -> Change runtime type -> T4 GPU).

In [ ]:
!nvidia-smi

## 📦 Step 2: Clone AquaYOLO Repository & Install Dependencies

In [ ]:
# Clone repository from GitHub
!git clone https://github.com/Aravind8833/AquaYOLO.git
%cd /content/AquaYOLO

# Install dependencies
!pip install -q ultralytics opencv-python matplotlib seaborn pyyaml pandas pillow tqdm kagglehub roboflow
print('\n Environment ready!')

## ⚡ Step 3: Direct Fast Download from Kaggle & Roboflow

Instead of waiting hours uploading gigabytes from home to Google Drive, Colab can download them in **under 1–2 minutes** at 100+ MB/s!

### 🔑 Kaggle Authentication (Takes 10 seconds):
1. Go to [kaggle.com](https://www.kaggle.com/) -> Profile icon (top right) -> **Settings**.
2. Scroll to the **API** section and click **Create New Token** (downloads `kaggle.json`).
3. Enter your **Kaggle Username** and **Key** below (or upload `kaggle.json`).

In [ ]:
import os
from pathlib import Path
import kagglehub

# =====================================================================
# ⚙️ OPTION 1: KAGGLE CREDENTIALS (ENTER YOUR USERNAME & KEY)
# =====================================================================
# Open your downloaded kaggle.json in notepad and paste username & key here:
os.environ["KAGGLE_USERNAME"] = "your_kaggle_username"
os.environ["KAGGLE_KEY"] = "your_kaggle_api_key"

# Or upload kaggle.json directly if you prefer:
if not os.path.exists(os.path.expanduser("~/.kaggle/kaggle.json")) and os.environ["KAGGLE_USERNAME"] == "your_kaggle_username":
    print("Upload your kaggle.json file:")
    from google.colab import files
    uploaded = files.upload()
    if uploaded:
        os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
        for fn in uploaded:
            os.rename(fn, os.path.expanduser(f"~/.kaggle/{fn}"))
        os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)

# 1. Download Kaggle Dataset 1: aquarium-data-cots
print("\n1. Downloading slavkoprytula/aquarium-data-cots from Kaggle...")
path_cots = kagglehub.dataset_download("slavkoprytula/aquarium-data-cots")
print(f"   COTS downloaded to: {path_cots}")

# 2. Download Kaggle Dataset 2: sea-animals-image-dataste
print("\n2. Downloading vencerlanz09/sea-animals-image-dataste from Kaggle...")
path_sea = kagglehub.dataset_download("vencerlanz09/sea-animals-image-dataste")
print(f"   Sea Animals downloaded to: {path_sea}")

### 🐠 Download Roboflow Dataset (`marine-life-pzpu1`)

For your Roboflow dataset (`https://universe.roboflow.com/boats-hr2s2/marine-life-pzpu1`):
- On the dataset page, click **Download Dataset** -> Select **YOLOv8** -> Choose **Show Code**.
- Copy and paste either your **Roboflow API Key** OR the **curl download command** below.

In [ ]:
import subprocess
from pathlib import Path

ROBOFLOW_DOWNLOAD_DIR = Path("/content/downloads/marine_life_roboflow")
ROBOFLOW_DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

# =====================================================================
# ⚙️ PASTE YOUR ROBOFLOW DOWNLOAD CURL COMMAND OR API KEY HERE:
# =====================================================================
# Method A: If you have your Roboflow API key:
ROBOFLOW_API_KEY = ""  # e.g. "abc123xyz"

# Method B: Or if you have the curl download URL snippet:
ROBOFLOW_CURL_URL = ""  # e.g. "https://universe.roboflow.com/ds/..."
# =====================================================================

path_roboflow = None

if ROBOFLOW_API_KEY:
    from roboflow import Roboflow
    print("Downloading from Roboflow via Python API...")
    rf = Roboflow(api_key=ROBOFLOW_API_KEY)
    project = rf.workspace("boats-hr2s2").project("marine-life-pzpu1")
    dataset = project.version(1).download("yolov8", location=str(ROBOFLOW_DOWNLOAD_DIR))
    path_roboflow = ROBOFLOW_DOWNLOAD_DIR
    print(f" Roboflow dataset downloaded to: {path_roboflow}")
elif ROBOFLOW_CURL_URL:
    print("Downloading from Roboflow via curl...")
    zip_dest = ROBOFLOW_DOWNLOAD_DIR / "roboflow.zip"
    subprocess.run(f"curl -L '{ROBOFLOW_CURL_URL}' > '{zip_dest}'", shell=True, check=True)
    subprocess.run(f"unzip -q '{zip_dest}' -d '{ROBOFLOW_DOWNLOAD_DIR}'", shell=True, check=True)
    path_roboflow = ROBOFLOW_DOWNLOAD_DIR
    print(f" Roboflow dataset downloaded and unzipped to: {path_roboflow}")
else:
    print("ℹ️ Note: If you don't have the Roboflow snippet right now, training will proceed using the two Kaggle datasets!")

## 🔄 Step 4: Harmonize & Merge Datasets (Zero Class Collisions)

Runs `dataset_merger.py` to:
- Remap conflicting class IDs from all sources into a unified marine taxonomy.
- Convert classification images (`sea-animals-image-dataste`) into valid detection targets.
- Build clean `/content/dataset` with balanced train, val, test splits and an updated `data.yaml`.

In [ ]:
from dataset_merger import merge_datasets

# Collect all downloaded dataset directories
downloaded_sources = [p for p in [path_cots, path_sea, path_roboflow] if p is not None and Path(p).exists()]

print(f"Datasets ready for harmonization: {len(downloaded_sources)}")
for s in downloaded_sources:
    print(f"  📂 {Path(s).name}")

LOCAL_DATASET_DIR = Path("/content/dataset")
TEMP_EXTRACT_DIR = Path("/content/temp_extract")

# Run merger
DATA_YAML_PATH = merge_datasets(downloaded_sources, LOCAL_DATASET_DIR, TEMP_EXTRACT_DIR)
print(f"\n Unified dataset ready: {DATA_YAML_PATH}")

## 🏋️ Step 5: High-Accuracy YOLO Training
Trains YOLOv8 with transfer learning, cosine learning rate decay, and mosaic augmentation to achieve high accuracy (>75% mAP50).

In [ ]:
# Run high-accuracy training
!python train_yolo.py \
    --data "{DATA_YAML_PATH}" \
    --model yolov8s.pt \
    --epochs 50 \
    --batch 16 \
    --imgsz 640 \
    --project /content/AquaYOLO/runs/yolo_train \
    --name colab_run

## 📊 Step 6: Display All Generated Metric Graphs
Displays every metric graph generated by YOLO directly in the notebook output:
1. **Training & Validation Loss / Metrics Curve** (`results.png`)
2. **Confusion Matrix & Normalized Confusion Matrix** (`confusion_matrix.png`)
3. **Precision-Recall Curve** (`PR_curve.png`)
4. **F1-Confidence Curve** (`F1_curve.png`)
5. **Precision & Recall Curves** (`P_curve.png`, `R_curve.png`)
6. **Validation Ground Truth vs Model Predictions** (`val_batch0_labels.jpg`, `val_batch0_pred.jpg`)

In [ ]:
from IPython.display import Image, display
from pathlib import Path
import pandas as pd

RUN_DIR = Path("/content/AquaYOLO/runs/yolo_train/colab_run")

def show_graph(image_filename, title):
    p = RUN_DIR / image_filename
    if p.exists():
        print("=" * 75)
        print(f"{title}: {p.name}")
        print("=" * 75)
        display(Image(filename=str(p), width=900))
    else:
        print(f"{image_filename} not generated yet.")

# 1. Training & Validation Results (Losses, Precision, Recall, mAP50, mAP50-95)
show_graph("results.png", "TRAINING & VALIDATION METRICS OVER EPOCHS")

# 2. Precision-Recall Curve
show_graph("PR_curve.png", "PRECISION-RECALL (PR) CURVE")

# 3. F1-Confidence Curve
show_graph("F1_curve.png", "F1-SCORE VS CONFIDENCE CURVE")

# 4. Precision & Recall vs Confidence Curves
show_graph("P_curve.png", "PRECISION VS CONFIDENCE CURVE")
show_graph("R_curve.png", "RECALL VS CONFIDENCE CURVE")

# 5. Confusion Matrices
show_graph("confusion_matrix.png", "CONFUSION MATRIX")
show_graph("confusion_matrix_normalized.png", "NORMALIZED CONFUSION MATRIX")

# 6. Visual Detections on Validation Set (Ground Truth vs Model Prediction)
show_graph("val_batch0_labels.jpg", "GROUND TRUTH LABELS")
show_graph("val_batch0_pred.jpg", "MODEL PREDICTIONS ON VALIDATION BATCH")

## 📋 Step 7: View Numerical Metrics Table (Epoch by Epoch)

In [ ]:
results_csv = RUN_DIR / "results.csv"
if results_csv.exists():
    df = pd.read_csv(results_csv)
    df.columns = [c.strip() for c in df.columns]
    print("\n--- FINAL EPOCHS SUMMARY ---")
    display(df.tail(10))
else:
    print("results.csv not found.")

## 🧠 Step 8: (Optional) Train Custom AquaYOLO Architecture
AquaYOLO model with CSPDarknet backbone, Spectral Attenuation Compensator (SAC), and Turbidity Contrast Gate (TCG).

In [ ]:
# Train custom AquaYOLO model
!python src/aquayolo/train.py \
    --data "{DATA_YAML_PATH.parent}" \
    --epochs 30 \
    --batch 4 \
    --size 640 \
    --lr 0.0001 \
    --out ./runs/aquayolo_custom_colab

# Show AquaYOLO loss curves
aqua_loss_plot = Path("./runs/aquayolo_custom_colab/loss_curve.png")
if aqua_loss_plot.exists():
    display(Image(filename=str(aqua_loss_plot), width=800))
aqua_comp_plot = Path("./runs/aquayolo_custom_colab/loss_components.png")
if aqua_comp_plot.exists():
    display(Image(filename=str(aqua_comp_plot), width=900))